# 14 - Data pipeline (Apache Beam): raw -> clean
Runs `pipeline/tennis_pipeline.py`, the ETL version of notebooks 04 and 05: it validates and types every raw match, quarantines bad rows with a reason, sets out-of-scope rows aside, works out each event's format and the window of possible match days, and summarises each tournament edition.
The SQL in notebooks 04-05 (the ELT path) is the independent check: both must give the same result.

| Part | What it does | Cost |
|---|---|---|
| 1-2 | Install, settings, unit tests | free |
| 3 | Copy the raw file from the bucket and check its checksum | free |
| 4 | Run on a small sample with deliberately bad rows; predict the counters first | free |
| 5 | Run on the full raw file on this machine (DirectRunner) | free |
| 6 | Reconcile the local result with the BigQuery tables from notebooks 04-07 | free |
| **7** | **Run on Google Cloud Dataflow** | **paid: only when `RUN_ON_DATAFLOW = True`** |
| 8-9 | Reconcile in BigQuery, read the quarantine | free (small queries) |
| 10 | Clean up: cancel any running Dataflow job, delete temporary files | free |

> **Safety:** nothing in this notebook starts Dataflow unless you set `RUN_ON_DATAFLOW = True` in Part 7. The pipeline itself also refuses to start on Dataflow without a cap of 1-2 workers.

## 1. Install
Sets the code version, clones the repository at that tag (the pipeline code and `provenance.py` come from there) and installs the pinned pipeline libraries. Safe to re-run: it only clones when the copy is missing or from another tag. If Colab asks to restart the session: restart, run this cell again, then continue from **Settings**.

In [ ]:
import os, subprocess, sys

# The code tag must contain the pipeline/ folder and this notebook. Create it on GitHub before running.
CODE_VERSION = "code-v4"
REPO = "katiedevane/tennis-project"
REPO_DIR = os.path.abspath("tennis-project")

def step(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        raise RuntimeError(f"Failed: {cmd}\n{r.stdout}\n{r.stderr}")
    return r.stdout

def copy_tag():
    r = subprocess.run(f"git -C {REPO_DIR} describe --tags", shell=True, text=True, capture_output=True)
    return r.stdout.strip() if r.returncode == 0 else None

if copy_tag() != CODE_VERSION:                     # missing, or a copy of another tag: (re)clone
    step(f"rm -rf {REPO_DIR}")
    step(f"git clone -q --depth 1 --branch {CODE_VERSION} https://github.com/{REPO}.git {REPO_DIR}")
assert copy_tag() == CODE_VERSION, "The clone is not at the expected tag"
assert os.path.exists(f"{REPO_DIR}/pipeline/requirements.txt"), "pipeline/ is not in this tag"
print("code:", CODE_VERSION, "->", REPO_DIR)

step(f"{sys.executable} -m pip install -q -r {REPO_DIR}/pipeline/requirements.txt")
print("pipeline libraries installed")

## Settings
Enable Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed

In [ ]:
import sys
for path in (REPO_DIR, f"{REPO_DIR}/pipeline"):
    if path not in sys.path:
        sys.path.insert(0, path)

from provenance import start_run, register, table_version, sha256, utc_stamp, SEED
NOTEBOOK = "14_pipeline"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Helpers

In [ ]:
import glob, gzip, io, csv, json, os, random, re, subprocess
import pandas as pd
import apache_beam as beam
from google.cloud import bigquery, storage

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
PIPE = f"{REPO_DIR}/pipeline"
DATAFLOW_REGION = "europe-west1"          # a region inside the EU multi-region of the bucket and datasets
PIPELINE_DATASET = "tennis_pipeline"      # the pipeline's own dataset: notebooks 04-10 tables are not touched

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

def sh(cmd, cwd=None):
    """Run a shell command, show its output as it comes, stop on failure. Returns the output."""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = []
    for line in p.stdout:
        print(line, end="")
        out.append(line)
    if p.wait():
        raise RuntimeError(f"Command failed (exit code {p.returncode})")
    return "".join(out)

def counters_from(output):
    """The counters the pipeline prints at the end, as a dict."""
    block = output.split("Counters:", 1)[-1]
    return {m.group(1): int(m.group(2).replace(",", "")) for m in re.finditer(r"^\s+(\w+)\s+([\d,]+)$", block, re.M)}

def read_jsonl(prefix):
    files = sorted(glob.glob(f"{prefix}-*.jsonl"))
    return pd.DataFrame([json.loads(l) for f in files for l in open(f)])

print("Apache Beam", beam.__version__, "| Python", sys.version.split()[0])

## 2. Unit tests
Small known inputs, checked with Beam's `TestPipeline`. Expected: `9 passed`. If a test fails, the pipeline is not ready: stop here.

In [ ]:
sh(f"{sys.executable} -m pytest -q test_tennis_pipeline.py", cwd=PIPE);

## 3. Get the raw file
The compressed raw file named in the latest ingest manifest (notebook 01), copied to this machine and checked against the recorded checksum.

In [ ]:
names = sorted(b.name for b in bucket.list_blobs(prefix="manifests/wta_matches_"))
manifest = json.loads(bucket.blob(names[-1]).download_as_text())
RAW = manifest["files"][0]
os.makedirs("data/raw", exist_ok=True)
RAW_LOCAL = "data/raw/" + RAW["file"].rsplit("/", 1)[1]
bucket.blob(RAW["file"]).download_to_filename(RAW_LOCAL)
assert sha256(open(RAW_LOCAL, "rb").read()) == RAW["sha256"], "Checksum differs from the manifest: stop and investigate"
print(RAW_LOCAL, "| checksum OK |", f"{RAW['rows']:,} rows recorded")

## 4. A small sample with deliberately bad rows
300 random raw rows (seeded), plus five rows made from one of them:
- empty `winner_id`, an impossible date, `w_ace = "many"`, `w_df = -1`: these break the contract, so they should be **quarantined**
- `minutes = 2475`: odd but possible, so it should be **kept and flagged**

**Predict the counters first** (with pandas, independently of the pipeline), then run the pipeline and compare.

In [ ]:
text = gzip.decompress(open(RAW_LOCAL, "rb").read()).decode("utf-8-sig")
header, *body = text.splitlines()
random.seed(SEED)
picked = list(csv.DictReader(io.StringIO("\n".join([header] + random.sample(body, 300)))))

rr_ids = {r["tourney_id"] for r in picked if r["round"] == "RR"}
in_scope = lambda r: r["tourney_level"] in ("G", "PM", "P", "I") and r["tourney_id"] not in rr_ids
template = next(r for r in picked if in_scope(r) and r["round"] in ("R32", "R16", "QF"))
crafted = [dict(template, match_num="9001", winner_id=""),
           dict(template, match_num="9002", tourney_date="20191345"),
           dict(template, match_num="9003", w_ace="many"),
           dict(template, match_num="9004", w_df="-1"),
           dict(template, match_num="9005", minutes="2475")]
sample = picked + crafted

os.makedirs("data/sample", exist_ok=True)
buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=header.split(","), lineterminator="\n")
writer.writeheader(); writer.writerows(sample)
with open("data/sample/sample.csv.gz", "wb") as f:
    f.write(gzip.compress(buf.getvalue().encode()))

predicted = {"rows_read": len(sample),
             "excluded_rows": sum(not in_scope(r) for r in sample),
             "bad_rows": 4,
             "good_rows": sum(in_scope(r) for r in sample) - 4}
print("predicted:", predicted)

In [ ]:
out = sh(f"{sys.executable} {PIPE}/tennis_pipeline.py --runner DirectRunner "
         f"--input 'data/sample/*.csv.gz' --output out/sample/run")
got = counters_from(out)
for k, v in predicted.items():
    print("PASS       " if got.get(k) == v else "CHECK AGAIN", f"{k}: predicted {v}, pipeline {got.get(k)}")
print("flag_minutes at least 1:", "PASS" if got.get("flag_minutes", 0) >= 1 else "CHECK AGAIN")
print(read_jsonl("out/sample/run_bad_rows")[["reason", "source", "line"]])

## 5. Full run on this machine (DirectRunner, free)
The whole raw file. Takes a few minutes. The counters must reconcile: `rows_read = excluded + bad + good`.

In [ ]:
out = sh(f"{sys.executable} {PIPE}/tennis_pipeline.py --runner DirectRunner "
         f"--input 'data/raw/*.csv.gz' --output out/full/run")
local_counters = counters_from(out)
assert local_counters["rows_read"] == RAW["rows"], "The pipeline did not read every row in the manifest"
assert local_counters["rows_read"] == sum(local_counters[k] for k in ("excluded_rows", "bad_rows", "good_rows"))
print("\nrows_read matches the manifest:", f"{RAW['rows']:,}")

In [ ]:
local = {t: read_jsonl(f"out/full/run_{t}") for t in ["matches_dated", "bad_rows", "excluded_rows", "edition_summary"]}
for t, d in local.items():
    print(f"{t:16s} {len(d):>7,} rows")
print("\nexcluded, by reason:\n", local["excluded_rows"]["reason"].value_counts().to_string())
if len(local["bad_rows"]):
    print("\nquarantined, by reason:\n", local["bad_rows"]["reason"].value_counts().to_string())

## 6. Reconciliation: does the Beam result (ETL) agree with the SQL result (ELT)?
Two independent methods on the same raw file. Every match and every shared column is compared, and the edition summary is compared with `tournament_editions` from notebook 07.

In [ ]:
def norm(v):
    """One representation for comparing: missing -> None, numbers -> float, dates -> 'YYYY-MM-DD'."""
    if v is None or v is pd.NA or (isinstance(v, float) and v != v):
        return None
    if isinstance(v, bool):
        return v
    if hasattr(v, "isoformat"):
        return v.isoformat()[:10]
    if isinstance(v, (int, float)) or type(v).__module__ == "numpy":
        return float(v)
    return v

etl = local["matches_dated"].set_index("match_id")
elt = run(f"SELECT * FROM `{PROJECT_ID}.tennis_clean.matches_dated`").set_index("match_id")
print("rows: ETL", len(etl), "| ELT", len(elt), "| same match_ids:", set(etl.index) == set(elt.index))

shared = [c for c in elt.columns if c in etl.columns]
common_ids = etl.index.intersection(elt.index)
etl, elt = etl.loc[common_ids], elt.loc[common_ids]
diff = {c: sum(norm(a) != norm(b) for a, b in zip(etl[c].tolist(), elt[c].tolist())) for c in shared}
diff = {c: n for c, n in diff.items() if n}
print("columns compared:", len(shared), "| columns that differ:", diff or "none")

In [ ]:
eds = local["edition_summary"].set_index("tourney_id")
ref = run(f"SELECT tourney_id, n_matches, event_format, start_date, last_match_day FROM `{PROJECT_ID}.tennis_clean.tournament_editions`").set_index("tourney_id")
print("editions: ETL", len(eds), "| ELT", len(ref), "| same ids:", set(eds.index) == set(ref.index))
ids = eds.index.intersection(ref.index)
eds, ref = eds.loc[ids], ref.loc[ids]
for a, b in [("matches", "n_matches"), ("event_format", "event_format"), ("start_date", "start_date"), ("last_match_day", "last_match_day")]:
    n = sum(norm(x) != norm(y) for x, y in zip(eds[a].tolist(), ref[b].tolist()))
    print(f"{a:15s} differs in {n} editions")

raw_rows = local_counters["rows_read"]
print("\nexcluded + bad = raw - clean:", local_counters["excluded_rows"] + local_counters["bad_rows"], "=", raw_rows - len(elt))

## 7. Run on Google Cloud Dataflow (paid)
**Leave `RUN_ON_DATAFLOW = False` until Parts 2-6 all pass.** Then set it to `True` and run this cell once.
- Batch job: it ends by itself (about 5-10 minutes, mostly start-up).
- At most 2 workers, small machines (`e2-standard-2`), one region (`europe-west1`).
- Writes four tables to the separate dataset `tennis_pipeline` and replaces them on every run (no duplicates).
- Afterwards, set it back to `False`, run Part 10, and note the job's cost (Dataflow > Jobs > your job > Cost).

In [ ]:
RUN_ON_DATAFLOW = False      # True only when you are ready: this starts a paid job

if not RUN_ON_DATAFLOW:
    print("Dataflow NOT started. Set RUN_ON_DATAFLOW = True in this cell when you are ready.")
else:
    # One-time setup (safe to repeat): services, the worker account's roles, the output dataset
    sh("gcloud services enable dataflow.googleapis.com compute.googleapis.com")
    number = sh(f"gcloud projects describe {PROJECT_ID} --format='value(projectNumber)'").strip().splitlines()[-1]
    worker = f"{number}-compute@developer.gserviceaccount.com"
    for role in ["roles/dataflow.worker", "roles/storage.objectAdmin", "roles/bigquery.dataEditor", "roles/bigquery.jobUser"]:
        sh(f"gcloud projects add-iam-policy-binding {PROJECT_ID} --member=serviceAccount:{worker} "
           f"--role={role} --condition=None --quiet > /dev/null")
    ds = bigquery.Dataset(f"{PROJECT_ID}.{PIPELINE_DATASET}")
    ds.location = REGION
    ds.labels = {"project": "tennis-ml"}
    bq.create_dataset(ds, exists_ok=True)

    job_name = "tennis-" + utc_stamp().lower().replace("t", "-").rstrip("z")
    out = sh(f"{sys.executable} {PIPE}/tennis_pipeline.py "
             f"--runner DataflowRunner --project {PROJECT_ID} --region {DATAFLOW_REGION} "
             f"--temp_location gs://{BUCKET_NAME}/dataflow/tmp --staging_location gs://{BUCKET_NAME}/dataflow/staging "
             f"--job_name {job_name} --max_num_workers 2 --machine_type e2-standard-2 "
             f"--input gs://{BUCKET_NAME}/{RAW['file']} --bq_dataset {PIPELINE_DATASET}")
    dataflow_counters = counters_from(out)
    print("\njob:", job_name, "| counters equal the local run:", dataflow_counters == local_counters)

In [ ]:
# After a successful Dataflow run: record the four tables in the index of data versions
if RUN_ON_DATAFLOW:
    outputs = [table_version(bq, f"{PROJECT_ID}.{PIPELINE_DATASET}.{t}")
               for t in ["matches_dated", "bad_rows", "excluded_rows", "edition_summary"]]
    register(bucket, f"{PIPELINE_DATASET} (Beam on Dataflow)", job_name, outputs=outputs,
             parents=[{"file": RAW["file"], "sha256": RAW["sha256"]}])

## 8. Reconciliation in BigQuery: Dataflow output (ETL) against notebooks 04-05 (ELT)
Runs only once the Dataflow tables exist. `matching_rows` must equal both row counts.

In [ ]:
from google.api_core.exceptions import NotFound
ETL = f"{PROJECT_ID}.{PIPELINE_DATASET}.matches_dated"
ELT = f"{PROJECT_ID}.tennis_clean.matches_dated"
try:
    etl_cols = {f.name for f in bq.get_table(ETL).schema}
except NotFound:
    etl_cols = None
    print("No Dataflow output yet: run Part 7 first.")

if etl_cols:
    shared = [f.name for f in bq.get_table(ELT).schema if f.name in etl_cols and f.name != "match_id"]
    same = " AND ".join(f"e.{c} IS NOT DISTINCT FROM l.{c}" for c in shared)
    print(run(f"""
SELECT (SELECT COUNT(*) FROM `{ETL}`) AS etl_rows,
       (SELECT COUNT(*) FROM `{ELT}`) AS elt_rows,
       COUNT(*) AS joined_rows,
       COUNTIF({same}) AS matching_rows,
       {len(shared)} AS columns_compared
FROM `{ETL}` e JOIN `{ELT}` l USING (match_id)""").T)

## 9. What did the quarantine and the scope filter catch?

In [ ]:
if etl_cols:
    print(run(f"SELECT source, reason, COUNT(*) AS bad_rows FROM `{PROJECT_ID}.{PIPELINE_DATASET}.bad_rows` GROUP BY 1, 2 ORDER BY 3 DESC"))
    print(run(f"SELECT reason, COUNT(*) AS rows_excluded, COUNT(DISTINCT tourney_id) AS editions FROM `{PROJECT_ID}.{PIPELINE_DATASET}.excluded_rows` GROUP BY 1 ORDER BY 2 DESC"))
    print(run(f"""SELECT match_id, tourney_name, round, score, minutes FROM `{ETL}`
                  WHERE flag_minutes ORDER BY minutes"""))
    print(run(f"SELECT COUNTIF(flag_serve_stats) AS flagged_serve_stats, COUNTIF(flag_minutes) AS flagged_minutes FROM `{ETL}`"))
else:
    print("Local results (Part 5):")
    print(local["matches_dated"].query("flag_minutes")[["match_id", "tourney_name", "round", "score", "minutes"]].sort_values("minutes"))

## 10. Clean up (always safe to run)
Cancels any Dataflow job still running in the region and deletes the pipeline's temporary and staging files. The result tables in `tennis_pipeline` are kept.

In [ ]:
listing = subprocess.run(f"gcloud dataflow jobs list --region={DATAFLOW_REGION} --status=active --format='value(JOB_ID)'",
                         shell=True, text=True, capture_output=True)
if listing.returncode:
    print("Dataflow is not enabled in this project (or cannot be listed): no job can be running.")
    active = []
else:
    active = listing.stdout.split()
for job in active:
    sh(f"gcloud dataflow jobs cancel {job} --region={DATAFLOW_REGION}")
print("active jobs cancelled:", len(active))

leftovers = list(bucket.list_blobs(prefix="dataflow/"))
for b in leftovers:
    b.delete()
print("temporary files deleted:", len(leftovers))
if not listing.returncode:
    sh(f"gcloud dataflow jobs list --region={DATAFLOW_REGION} --status=active")      # should list nothing